# Finetuning Granite Speech (LoRA)


In [1]:
!pip install -U "transformers>=4.52" accelerate torch torchaudio torchvision jiwer pandas tqdm torchcodec peft bitsandbytes


Looking in indexes: https://nexus.iisys.de/repository/ki-awz-pypi-group/simple, https://pypi.org/simple


## Setup

In [2]:
import os
import torch
from pathlib import Path
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

MODEL_ID = "ibm-granite/granite-speech-5.0-2b"

MANIFEST_DIR = Path("asr_finetune")
OUTPUT_DIR = MANIFEST_DIR / "checkpoint"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_ID, dtype=torch.bfloat16).to("cuda:0")
print(f"Model loaded on {model.device} with dtype {model.dtype}")

Loading weights:   0%|          | 0/954 [00:00<?, ?it/s]

Model loaded on cuda:0 with dtype torch.bfloat16


## Load the Training and Dev Manifests

In [3]:
import json

MANIFEST_TRAIN = MANIFEST_DIR / "manifest_train_new.jsonl"
MANIFEST_DEV = MANIFEST_DIR / "manifest_dev_new.jsonl"

def read_manifest(path):
    with open(path, encoding="UTF-8") as rf:
        return [json.loads(line) for line in rf]

train_data = read_manifest(MANIFEST_TRAIN)[:2000]
dev_data = read_manifest(MANIFEST_DEV)[:100]

print(len(train_data), "train samples,", len(dev_data), "dev samples")
print(train_data[0])

2000 train samples, 100 dev samples
{'audio': '/training-1/asr/voxpopuli/out/transcribed_data/en/2013/20131007-0900-PLENARY-19-en_20131007-21:26:04_7.wav', 'prompt': 'Transcribe the text, put named entities into brackets.', 'text': 'this could be done by looking more specifically into the suggested fallback clause in [article] [twenty] [one] in cases brought by the employee against the employer defining as relevant the place of business from which the employee receives or received day to day instructions'}


## Prompt Construction

In [4]:
import torchaudio

def load_audio(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)

    waveform = torchaudio.functional.resample(
        waveform, sample_rate, 16000
    )

    waveform = waveform.mean(dim=0)
    return waveform

def build_chat_prompt(user_prompt):
    chat = [
        {"role": "system", "content": "You are Granite, developed by IBM. You are a helpful AI assistant."},
        {"role": "user", "content": f"<|audio|>{user_prompt}"},
    ]
    return processor.tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

DEFAULT_PROMPT = "Transcribe the text, put named entities into brackets."

def build_user_prompt(sample):
    return DEFAULT_PROMPT

## LoRA Configuration (PEFT)

In [5]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,259,840 || all params: 2,317,401,436 || trainable%: 0.1838


## Collate Function (Audio + Prompt + Target Text)

In [6]:
def collate_fn(batch):
    wavs = [load_audio(sample["audio"]) for sample in batch]
    prompts = [build_chat_prompt(build_user_prompt(sample)) for sample in batch]
    inputs = processor(prompts, wavs, return_tensors="pt", padding=True)

    eos_id = processor.tokenizer.eos_token_id   # the model must learn to stop
    target_ids = []
    for sample in batch:
        t = processor.tokenizer(sample["text"], add_special_tokens=False,
                                return_tensors="pt").input_ids[0]
        target_ids.append(torch.cat([t, torch.tensor([eos_id], dtype=t.dtype)]))
    batch_size, prompt_len = inputs["input_ids"].shape
    max_len = prompt_len + max(len(t) for t in target_ids)
    pad_id = processor.tokenizer.pad_token_id if processor.tokenizer.pad_token_id is not None else 0

    input_ids = torch.full((batch_size, max_len), pad_id, dtype=inputs["input_ids"].dtype)
    attention_mask = torch.zeros((batch_size, max_len), dtype=inputs["attention_mask"].dtype)
    labels = torch.full((batch_size, max_len), -100, dtype=torch.long)

    input_ids[:, :prompt_len] = inputs["input_ids"]
    attention_mask[:, :prompt_len] = inputs["attention_mask"]
    for i, t in enumerate(target_ids):
        input_ids[i, prompt_len:prompt_len + len(t)] = t
        attention_mask[i, prompt_len:prompt_len + len(t)] = 1
        labels[i, prompt_len:prompt_len + len(t)] = t

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        **{k: v for k, v in inputs.items() if k not in ("input_ids", "attention_mask")},
    }

batch = collate_fn(train_data[:2])
mask = batch["labels"] != -100
print(processor.tokenizer.decode(batch["labels"][0][mask[0]]))

this could be done by looking more specifically into the suggested fallback clause in [article] [twenty] [one] in cases brought by the employee against the employer defining as relevant the place of business from which the employee receives or received day to day instructions<|end_of_text|>


## Training Configuration

In [7]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR.as_posix(),
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    num_train_epochs=2,
    eval_strategy="epoch",
    logging_steps=10,
    logging_first_step=True,
    save_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    report_to=[],
    remove_unused_columns=False,
    disable_tqdm=False,
)


## Evaluation (Dev WER)

In [8]:
import jiwer
from tqdm import tqdm
import re
import string
symbols = '!"#$%&\'()*+,-./:;<=>?@^_`{|}~'
def normalize(text):
    text = text.translate(str.maketrans("", "", symbols)).lower().strip()
    text = re.sub(r" +", " ", text).strip()
    return text
    
def compute_dev_wer():
    model.eval()
    wers = []
    for sample in tqdm(dev_data):
        wav = load_audio(sample["audio"])
        text_prompt = build_chat_prompt(build_user_prompt(sample))
        inputs = processor(text_prompt, wav, device=model.device, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=400, do_sample=False, num_beams=1,
                                 use_cache=True)
        hyp = processor.tokenizer.decode(
            out[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
        hyp = normalize(hyp)
        label = normalize(sample["text"])
        print("="*30)
        print(f"PRED:\t\t{hyp}")
        print(f"LABEL\t\t{label}")
        print("="*30)
        
        wers.append(jiwer.wer(label, hyp))
    model.train()
    return sum(wers) / len(wers)



In [9]:
print("Dev WER before training:", compute_dev_wer())

  1%|          | 1/100 [00:02<03:27,  2.10s/it]

PRED:		this could be done by looking more specifically into the suggested fallback clause in article 21 in cases brought by the employee against the employer defining as relevant the place of business from which the employee receives or received day to day instructions
LABEL		this could be done by looking more specifically into the suggested fallback clause in [article] [twenty] [one] in cases brought by the employee against the employer defining as relevant the place of business from which the employee receives or received day to day instructions


  2%|▏         | 2/100 [00:03<02:52,  1.76s/it]

PRED:		2012 for instance the new brussels i regulation provides the right for employees to sue several employers together and the right for employees to have access to court in europe even if the employer is domiciled outside europe the commission will
LABEL		[two] [thousand] [and] [twelve] for instance the new [brussels] [i] [regulation] provides for the right for employees to sue several employers together and the right for employees to have access to courts in [europe] even if the employer is domiciled outside [europe] the commission will


  3%|▎         | 3/100 [00:04<02:21,  1.46s/it]

PRED:		i ask you therefore to enforce the standing order and refer these reports back to committee so they do not take up our time when they should be sorted out in committee
LABEL		i ask you therefore to enforce the standing order and refer these reports back to committee so that they do not take up our time when they should be sorted out in committee


... (190 outputs left out)

 99%|█████████▉| 99/100 [01:42<00:00,  1.06it/s]

PRED:		as a consequence we cannot influence properly the world trade liberalisation programme
LABEL		as a consequence we can not influence properly the world trade liberalisation programme


100%|██████████| 100/100 [01:43<00:00,  1.03s/it]

PRED:		we cannot influence tariff reductions that would benefit the people of the united kingdom
LABEL		we can not influence tariff reductions that would benefit the people of [the] [united] [kingdom]
Dev WER before training: 0.11389096115420187


## Train

In [8]:
from transformers import Trainer, TrainerCallback


class ConsoleLogCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"step {state.global_step}: {logs}")


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=dev_data,
    data_collator=collate_fn,
    callbacks=[ConsoleLogCallback()],
)

trainer.train()


Epoch,Training Loss,Validation Loss
1,0.962633,0.357141
2,0.675128,0.271826


step 1: {'loss': 15.446884155273438, 'grad_norm': 28.56083106994629, 'learning_rate': 0.0001, 'epoch': 0.008}
step 10: {'loss': 11.454444037543404, 'grad_norm': 21.796422958374023, 'learning_rate': 9.64e-05, 'epoch': 0.08}
step 20: {'loss': 7.598732757568359, 'grad_norm': 14.037158966064453, 'learning_rate': 9.240000000000001e-05, 'epoch': 0.16}
step 30: {'loss': 3.2368389129638673, 'grad_norm': 6.24726676940918, 'learning_rate': 8.840000000000001e-05, 'epoch': 0.24}
step 40: {'loss': 1.8376964569091796, 'grad_norm': 7.663445949554443, 'learning_rate': 8.44e-05, 'epoch': 0.32}
step 50: {'loss': 1.5636030197143556, 'grad_norm': 5.199967384338379, 'learning_rate': 8.04e-05, 'epoch': 0.4}
step 60: {'loss': 1.4048994064331055, 'grad_norm': 5.692507266998291, 'learning_rate': 7.64e-05, 'epoch': 0.48}
step 70: {'loss': 1.276594352722168, 'grad_norm': 6.675459384918213, 'learning_rate': 7.24e-05, 'epoch': 0.56}
step 80: {'loss': 0.9559233665466309, 'grad_norm': 6.134491443634033, 'learning_ra

TrainOutput(global_step=250, training_loss=1.7023813610076903, metrics={'train_runtime': 460.6321, 'train_samples_per_second': 8.684, 'train_steps_per_second': 0.543, 'total_flos': 1.4875564319096256e+16, 'train_loss': 1.7023813610076903, 'epoch': 2.0})

In [9]:
print("Dev WER AFTER training:", compute_dev_wer())

  1%|          | 1/100 [00:12<20:39, 12.52s/it]

PRED:		this could be done by looking more specifically into the suggested fallback clause in [agre] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [t] [
LABEL		this could be done by looking more specifically into the suggested fallback clause in [article] [twenty] [one] in cases brought by the employee against the employer defining as relevant the place of business from which the employee receives or received day to day instructions


  2%|▏         | 2/100 [00:14<10:39,  6.53s/it]

PRED:		[two] [thousand] [and] [twelve] for instance the new [brussels] [i] [regulation] [foresees] the right for employees to sue several employers together and the right for employees to have access to court in [europe] even if the employer is domiciled outside [europe] the commission will
LABEL		[two] [thousand] [and] [twelve] for instance the new [brussels] [i] [regulation] provides for the right for employees to sue several employers together and the right for employees to have access to courts in [europe] even if the employer is domiciled outside [europe] the commission will


  3%|▎         | 3/100 [00:15<06:32,  4.04s/it]

PRED:		i ask you therefore to enforce the standing order and refer these reports back to committee so that they do not take up our time when they should be sorted out in committee
LABEL		i ask you therefore to enforce the standing order and refer these reports back to committee so that they do not take up our time when they should be sorted out in committee


... (186 outputs left out)

 99%|█████████▉| 99/100 [01:59<00:00,  1.04it/s]

PRED:		as a consequence we cannot influence properly the world trade liberalisation programme
LABEL		as a consequence we can not influence properly the world trade liberalisation programme


100%|██████████| 100/100 [02:00<00:00,  1.21s/it]

PRED:		we can not influence tariff reductions that would benefit the people of [the] [united] [kingdom]
LABEL		we can not influence tariff reductions that would benefit the people of [the] [united] [kingdom]
Dev WER AFTER training: 0.1113188302855292


## Write Out the Checkpoint (LoRA Merged into the Base Model)

In [ ]:
merged_model = model.merge_and_unload()   # merge LoRA weights into the base model
merged_model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print(f"Checkpoint written to {OUTPUT_DIR}")